In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
from astropy.io import fits
from spectractor.simulation.throughput import TelescopeTransmission
from spectractor.config import load_config
from spectractor.extractor.targets import Star
from spectractor.extractor.dispersers import Hologram
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from scipy.signal import convolve
from scipy.integrate import trapezoid

%matplotlib widget

In [ ]:
%matplotlib widget

In [ ]:
data_dir = '/home/lmousset/Projets_Recherche/LSST/Star_catalogue/'

In [ ]:
def plot_spectra(specs, colorparams):
    colormap = cm.Reds
    normalize = mcolors.Normalize(vmin=np.min(colorparams), vmax=np.max(colorparams))

    fig  = plt.figure(figsize=(7,3))
    for s, spec in enumerate(specs):
        plt.plot(spec[0, :], spec[1, :], color = colormap(normalize(colorparams[s])))
    plt.grid()
    plt.xlabel(r"$\lambda$ [nm]")
    plt.ylabel(f"Flux ")
    
    # Colorbar setup
    s_map = cm.ScalarMappable(norm=normalize, cmap=colormap)
    s_map.set_array(colorparams)

    # If color parameters is a linspace, we can set boundaries in this way
    #if len(colorparams) > 1:
        #halfdist = (colorparams[1] - colorparams[0])/2.0
        #boundaries = np.linspace(colorparams[0] - halfdist, colorparams[-1] + halfdist, len(colorparams) + 1)
    #boundaries = np.linspace(0.5, 2.5, len(colorparams) + 1)

    # Use this to emphasize the discrete color values
    cbar = fig.colorbar(s_map, ax=plt.gca()) #, spacing='proportional', ticks=colorparams, boundaries=boundaries, format='%2.2g') # format='%2i' for integer

    # Use this to show a continuous colorbar
    #cbar = fig.colorbar(s_map, spacing='proportional', ticks=colorparams, format='%2i')

    cbar.set_label(r"Airmass $z$")
    plt.tight_layout()
    return fig
"""
def plot_spectra(specs, colorparams):
    colormap = cm.Reds
    normalize = mcolors.Normalize(vmin=np.min(colorparams), vmax=np.max(colorparams))

    fig  = plt.figure(figsize=(7,3))
    for spec in specs:
        plt.plot(spec.lambdas, spec.data, color = colormap(normalize(spec.airmass)))
    plt.grid()
    plt.xlabel(r"$\lambda$ [nm]")
    plt.ylabel(f"Flux [{spec.units}]")
    
    # Colorbar setup
    s_map = cm.ScalarMappable(norm=normalize, cmap=colormap)
    s_map.set_array(colorparams)

    # If color parameters is a linspace, we can set boundaries in this way
    halfdist = (colorparams[1] - colorparams[0])/2.0
    boundaries = np.linspace(colorparams[0] - halfdist, colorparams[-1] + halfdist, len(colorparams) + 1)

    # Use this to emphasize the discrete color values
    cbar = fig.colorbar(s_map, ax=plt.gca()) #, spacing='proportional', ticks=colorparams, boundaries=boundaries, format='%2.2g') # format='%2i' for integer

    # Use this to show a continuous colorbar
    #cbar = fig.colorbar(s_map, spacing='proportional', ticks=colorparams, format='%2i')

    cbar.set_label(r"Airmass $z$")
    plt.show()
    return fig
"""

def polyfit_sigmaclip(x, y, yerr, deg=1, sigmaclip=5, niter=3):
    w = 1/yerr
    goods = (~np.isnan(y)) & (~np.isnan(yerr))
    w[~goods] = 0
    '''
    # unweighted to remove strong outliers with little error bars
    pval, pcov = np.polyfit(x[goods], y[goods], deg=deg, cov='unscaled')
    model = np.polyval(pval, x)
    clip = np.abs((y - model) * w) > sigmaclip
    if np.sum(~clip & goods) <= deg+1:
        return pval, pcov, (w==0.)
    w[clip | ~goods] = 0

    counter = np.sum(w>0) - (deg + 1)
    new_clip = True
    while new_clip and counter>0:
        worst = np.argmax(np.abs(((y - model) * w)))
        w[worst] = 0
        pval = np.polyfit(x[goods], y[goods], deg=deg, w=w[goods], cov=None)
        model = np.polyval(pval, x)
        clip = np.abs((y - model) * w) > sigmaclip
        new_clip = np.any(clip)
        counter -= 1
    '''
    # weighted fit on remining data points
    for n in range(niter):
        pval, pcov = np.polyfit(x[goods], y[goods], w=w[goods], deg=deg, cov='unscaled')
        model = np.polyval(pval, x)
        res = np.abs((y - model) * w)
        clip = res > sigmaclip
        if np.any(clip):
            worst = np.argmax(res)
            w[worst] = 0
        else:
            break
        #print(n, x[worst], y[worst], w[worst])
        #if np.sum(~clip & goods) <= deg+1:
        #    break
        #w[clip | ~goods] = 0
    return pval, pcov, (w==0.)

In [ ]:
# Load the dataframe
df = pd.read_parquet(data_dir + "auxtel_atmosphere_202311_v3_2_1_fixA2fixA1_RobustFit_newThroughputs.gz")

# Apply some cuts to select good quality spectra
print(len(df))
filtered = (df["CHI2_FIT"] < 200) 
print(np.sum(filtered))
filtered = filtered  & (df["PSF_REG"] > 1e-2) 
print(np.sum(filtered))
filtered = filtered  & (df["D2CCD"] > 186.0)  & (df["D2CCD"] < 188)  
print(np.sum(filtered))
filtered = filtered  & (df["PIXSHIFT"] > -1.)  & (df["PIXSHIFT"] < 1) 
print(np.sum(filtered))
filtered = filtered  & (df["alpha_0_2"] < 6)  & (df["alpha_0_1"] >1.2)  & (df["gamma_0_1"] < 20) # & (rec["gamma_0_2"] < 20)
print(np.sum(filtered))
filtered = filtered  & (df["PWV [mm]_err_rum"] > 0) & (df["PWV [mm]_err"] > 0) # & (rec["PWV [mm]_err"] < 5)
print(np.sum(filtered))


df_filtered = df[filtered]
print(df_filtered["id"])

In [ ]:
# Add a column for the night of observation
df_filtered["night"] = (df_filtered["id"] // 100_000).astype(int)
df_filtered["night"]

In [ ]:
df_filtered["AIRMASS"]

all_targets = np.unique(df_filtered["TARGET"]) # All stars that were observed

In [ ]:
df_filtered.head()

# Plot the spectra for one target and one filter

In [ ]:
print(all_targets)

In [ ]:
# Choose a target and a filter and the night range to explore
mytarget = ["HD2811", "HD200654"]
print(mytarget)
myfilter = ["empty"]

night_min = 20250902
night_max = 20251002

In [ ]:
# Loop over nights
for night, groupn in df_filtered.groupby("night"):
    if night < night_min:
        continue
    if night > night_max:
        break
    # Loop over filters
    for filt, groupf in groupn.groupby("FILTER"):
        if filt in myfilter:
            # Loop over targets
            for target, groupt in groupf.groupby(["TARGET"]):
                if target[0] in mytarget:
                    # Number of spectra for this target
                    nspecs = len(groupt["id"]) 
                    print(f"Target {target} with filter {filt} has {nspecs} spectra")
                    
                    # Get spectra and airmasses
                    specs = [np.load(data_dir + f"Saved_spectra/{night}/{filt}/{target[0]}/spec_visit{i:02d}.npy") for i in range(nspecs)]
                    airmasses = [groupt.iloc[i]["AIRMASS"] for i in range(nspecs)]
                    print(f'Airmasses: {airmasses}')
                    
                    # Plot
                    fig = plot_spectra(specs, airmasses)
                    fig.suptitle(f"Night {night} - Target {target[0]} with filter {filt}")

In [ ]:
wls_sed = np.arange(lambda_min, lambda_max, 1) # All wavelengths for the SED plots

fig = plt.figure()
for target in all_targets:
    s = Star(target)
    sed = s.sed(wls_sed)
    plt.plot(wls_sed, sed)


# Analysis

In [ ]:
all_targets

In [ ]:
# Choose a target and a filter and the night range to explore
mytarget = "HD185975"  # all_targets[11]
print(mytarget)
myfilter = "empty"

night_min = 20250902
night_max = 20251231

# Put all spectra and airmasses in a single list to plot them all together
all_specs = []
all_airmasses = []
# Loop over nights
for night, groupn in df_filtered.groupby("night"):
    if night < night_min:
        continue
    if night > night_max:
        break
    # Loop over filters
    for filt, groupf in groupn.groupby("FILTER"):
        if filt == myfilter:
            # Loop over targets
            for target, groupt in groupf.groupby(["TARGET"]):
                if target[0] == mytarget:
                    # Number of spectra for this target
                    nspecs = len(groupt["id"]) 
                    #print(f"Target {target} with filter {filt} has {nspecs} spectra")
                    
                    # Get spectra and airmasses
                    for i in range(nspecs):
                        spec = np.load(data_dir + f"Saved_spectra/{night}/{filt}/{target[0]}/spec_visit{i:02d}.npy")
                        airmass = groupt.iloc[i]["AIRMASS"]
                        all_specs.append(spec)
                        all_airmasses.append(airmass)
                    
all_airmasses = np.array(all_airmasses)

# Plot
fig = plot_spectra(all_specs, all_airmasses)
#fig.suptitle(f"Night {night} - Target {target[0]} with filter {filt}")

In [ ]:
# Define the wavelength range
if myfilter=="OG550_65mm_1":
    lambda_min, lambda_max = 550, 1050
elif myfilter=="BG40_65mm_1":
    lambda_min, lambda_max = 370, 650
else:
    lambda_min, lambda_max = 370, 1050

wls = np.arange(lambda_min, lambda_max, 10) # Wavelengths of interest
nwls = len(wls)
print(nwls)
wls_sed = np.arange(lambda_min, lambda_max, 1) # All wavelengths for the SED plots

In [ ]:
# Magnitudes and uncertainties on magnitudes at the wavelengths of interest

all_mags = []
all_dmags = []

for s, spec in enumerate(all_specs):
    airmass = all_airmasses[s]

    mags = np.zeros(nwls) # magnitudes
    dmags = np.zeros(nwls) # uncertainties on magnitudes
    for i, wl in enumerate(wls):
        mags[i] = -2.5*np.log10(np.interp(wl, spec[0, :], spec[1, :]))
        dmags[i] = 2.5/np.log(10) * np.abs(np.interp(wl, spec[0, :], spec[2, :])/np.interp(wl, spec[0, :], spec[1, :]))
    all_mags.append(mags)
    all_dmags.append(dmags)

all_mags = np.array(all_mags)
all_dmags = np.array(all_dmags)

In [ ]:
all_amplitudes = -2.5*np.log10([np.sum(spec[1, :]) for spec in all_specs])
all_amplitudes = np.array(all_amplitudes)

len(all_amplitudes*3.)

In [ ]:
# Cut the cloud presence looking at spectrum integral
sigmaclip = 1
niter = 10

# Amplitude of each spectrum
all_amplitudes = -2.5*np.log10([np.sum(spec[1, :]) for spec in all_specs])
yerr = 5e-5*all_amplitudes

pval, pcov, clipped = polyfit_sigmaclip(np.array(all_airmasses), np.array(all_amplitudes), deg=1, yerr=yerr, sigmaclip=sigmaclip, niter=niter)

fig = plt.figure()
plt.scatter(all_airmasses, all_amplitudes)
plt.scatter(all_airmasses[clipped], all_amplitudes[clipped], color="gray")
plt.plot(all_airmasses, np.polyval(pval, all_airmasses))
plt.xlabel("airmass")
plt.ylabel("Integral")
plt.grid()
plt.show()


all_dmags[clipped, :] = np.inf

# Plot the spectra after integral clipping
fig = plot_spectra([spec for k, spec in enumerate(all_specs) if not clipped[k]], all_airmasses[~clipped])


In [ ]:
# Airmass regression

all_pvals = []
all_pcovs = []

fig = plt.figure()
for k, wl in enumerate(wls):
    pval, pcov, clipped = polyfit_sigmaclip(all_airmasses, all_mags[:, k], deg=1, yerr=all_dmags[:, k], sigmaclip=sigmaclip, niter=niter)   
    all_pvals.append(pval)
    all_pcovs.append(pcov)

    p = plt.errorbar(all_airmasses, all_mags[:, k], yerr=all_dmags[:, k], marker='+', linestyle='none', label=wl)
    plt.plot(all_airmasses, np.polyval(pval, all_airmasses), color=p[0].get_color())
    plt.errorbar(all_airmasses[clipped], all_mags[:, k][clipped], yerr=all_dmags[:, k][clipped], marker='+', linestyle='none', color="gray", zorder=42)

plt.grid()
plt.legend()
plt.ylabel("Magnitude [mag]")
plt.xlabel("Airmass")
plt.show()

all_pvals = np.array(all_pvals)
all_pcovs = np.array(all_pcovs)


In [ ]:
# Residuals to airmass regression
fig = plt.figure()
for k, wl in enumerate(wls[::]):
    p = plt.errorbar(all_airmasses, all_mags[:, k] - np.polyval(all_pvals[k], all_airmasses), yerr=all_dmags[:, k], marker='+', linestyle='none', label=wl)
#plt.ylim(-2, 2)
plt.grid()
plt.xlabel("Airmass")
plt.ylabel("Residuals to airmass regression [mag]")
plt.legend()
plt.show()

In [ ]:
all_pvals.shape
all_pcovs.shape

In [ ]:

fig = plt.figure()
plt.errorbar(wls, all_pvals[:, 0], yerr=all_pcovs[:, 0, 0], marker="+", color="k", linestyle="none")
plt.xlabel(r"$\lambda$ [nm]")
plt.ylabel(r"$k$ [mag/airmass]")
plt.grid()
plt.show()


In [ ]:
mytarget = "HD185975"

In [ ]:
spec_toa = 10**(-0.4 * all_pvals[:, 1]) # Coeff directeur de la régression airmass-magnitude, converti en TOA
spec_toa_err = np.abs(-0.4 * np.log(10) * spec_toa * np.sqrt(all_pcovs[:, 1, 1]))

# SED of the target
s = Star(mytarget)
sed = s.sed(wls_sed)

# Load the disperser
h = Hologram("holo4_003")

# Detector gain
gain = 0.8

tel = TelescopeTransmission(filter_label=myfilter)

sed *= tel.transmission(wls_sed) * h.transmission(wls_sed) * gain


fig = plt.figure()
plt.errorbar(wls, spec_toa, yerr=spec_toa_err, marker="+", color="k", linestyle="none")
#plt.plot(wls_sed, sed)
plt.plot(wls_sed, sed*trapezoid(spec_toa, x=wls) / trapezoid(sed, x=wls_sed))
plt.xlabel(r"$\lambda$ [nm]")
plt.ylabel(f"Spectrum TOA with throughput")
plt.title(f"{mytarget}")
plt.grid()
plt.show()

In [ ]:
spec_toa = 10**(-0.4*np.array(all_pvals[:, 1])) / tel.transmission(wls) / h.transmission(wls) / gain
spec_toa_err = np.abs(-0.4 * np.log(10) * spec_toa * np.sqrt(all_pcovs[:, 1, 1]))


            
fig = plt.figure()
plt.errorbar(wls, spec_toa, yerr=spec_toa_err, marker="+", color="k", linestyle="-")
plt.plot(wls_sed, sed*trapezoid(y=spec_toa, x=wls)/trapezoid(y=sed, x=wls_sed))
plt.ylim(0, 1.2*np.max(spec_toa))
plt.xlabel(r"$\lambda$ [nm]")
plt.ylabel(f"Spectrum TOA")
plt.title(f"{mytarget}")
plt.grid()
plt.show()